# Sistem Peringatan Dini GSII Google Trends

Notebook ini mengolah data Google Trends untuk membentuk **Google Search Interest Index (GSII)**, membuat fitur time series, membandingkan model forecasting, dan menghasilkan status peringatan dini peningkatan minat pencarian judi online.

## 1. Instalasi Dependency

Jalankan cell ini kalau package belum tersedia. Jika sudah pernah install, cell ini boleh dilewati.

In [ ]:
# Jika belum terinstall, hapus tanda komentar di baris berikut:
# %pip install -r ../requirements.txt

# Opsional untuk model XGBoost dan LightGBM:
# %pip install -r ../requirements-optional.txt

## 2. Import Library dan Fungsi Pipeline

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from pipelines.google_trends_gsii_pipeline import (
    DEFAULT_INPUT,
    FIGURES_DIR,
    PROCESSED_DIR,
    TABLES_DIR,
    add_gsii,
    build_keyword_summary,
    create_early_warning,
    create_time_series_features,
    ensure_output_dirs,
    load_google_trends_csv,
    plot_correlation_heatmap,
    plot_keyword_trends,
    plot_predictions,
    train_and_evaluate,
)

sns.set_theme(style="whitegrid")
ensure_output_dirs()

DATA_PATH = PROJECT_ROOT / DEFAULT_INPUT
DATA_PATH

## 3. Load Dataset

Loader ini sudah menangani kemungkinan CSV Google Trends memiliki baris metadata di bagian atas.

In [ ]:
df = load_google_trends_csv(DATA_PATH)
df.head()

## 4. Informasi Dataset

In [ ]:
info_table = pd.DataFrame({
    "column": df.columns,
    "dtype": [df[col].dtype for col in df.columns],
    "missing_value": [df[col].isna().sum() for col in df.columns],
})

print(f"Jumlah baris: {len(df)}")
print(f"Jumlah kolom: {len(df.columns)}")
info_table

## 5. Statistik Deskriptif Keyword

Ringkasan ini memuat rata-rata, median, standar deviasi, nilai maksimum, jumlah nilai bukan nol, dan rekomendasi awal keyword untuk model.

In [ ]:
keyword_summary = build_keyword_summary(df)
keyword_summary.to_csv(PROJECT_ROOT / TABLES_DIR / "keyword_summary.csv", index_label="keyword")
keyword_summary

## 6. Visualisasi Tren Keyword

In [ ]:
keyword_columns = [col for col in df.columns if col != "date"]

plt.figure(figsize=(14, 7))
for col in keyword_columns:
    plt.plot(df["date"], df[col], linewidth=1.8, label=col)
plt.title("Tren Interest Over Time Keyword Judi Online")
plt.xlabel("Tanggal")
plt.ylabel("Interest")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

plot_keyword_trends(df)

## 7. Heatmap Korelasi Antar Keyword

In [ ]:
corr = df[keyword_columns].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="RdYlBu_r", center=0, fmt=".2f", linewidths=0.5)
plt.title("Heatmap Korelasi Antar Keyword")
plt.tight_layout()
plt.show()

plot_correlation_heatmap(df)

## 8. Membentuk Google Search Interest Index (GSII)

Rumus awal: rata-rata dari `slot_gacor`, `judi_slot`, dan `judi_online`. Jika salah satu kolom tidak tersedia, script otomatis memakai kolom utama yang tersedia.

In [ ]:
df_gsii, gsii_columns = add_gsii(df)
df_gsii.to_csv(PROJECT_ROOT / PROCESSED_DIR / "google_trends_gsii.csv", index=False)

print("Kolom pembentuk GSII:", gsii_columns)
df_gsii[["date", *gsii_columns, "GSII"]].tail(10)

## 9. Membuat Fitur Time Series

Target forecasting adalah `target_next_month = GSII.shift(-1)`.

In [ ]:
featured = create_time_series_features(df_gsii)
featured.to_csv(PROJECT_ROOT / PROCESSED_DIR / "google_trends_gsii_features.csv", index=False)
featured.head()

## 10. Split Data dan Training Model

Split menggunakan urutan waktu: 80% awal sebagai data latih, 20% terakhir sebagai data uji.

In [ ]:
evaluation, prediction_table, best_model = train_and_evaluate(featured)

evaluation.to_csv(PROJECT_ROOT / TABLES_DIR / "model_evaluation.csv", index=False)
prediction_table.to_csv(PROJECT_ROOT / TABLES_DIR / "model_predictions.csv", index=False)

print(f"Model terbaik berdasarkan RMSE: {best_model}")
evaluation

## 11. Visualisasi Aktual vs Prediksi

In [ ]:
plot_predictions(prediction_table, best_model)

model_columns = [col for col in prediction_table.columns if col.startswith("pred_")]
for col in model_columns:
    model_name = col.replace("pred_", "")
    plt.figure(figsize=(12, 5))
    plt.plot(prediction_table["date"], prediction_table["target_next_month"], marker="o", linewidth=2, label="Aktual bulan depan")
    plt.plot(prediction_table["date"], prediction_table[col], marker="s", linewidth=2, label=f"Prediksi {model_name}")
    plt.title(f"Aktual vs Prediksi GSII: {model_name}")
    plt.xlabel("Tanggal observasi")
    plt.ylabel("GSII bulan depan")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 12. Early Warning Rule

- Naik >= 20%: `Waspada`
- Naik >= 40%: `Tinggi`
- Naik positif tetapi < 20%: `Perhatian`
- Tidak naik: `Normal`

In [ ]:
warning = create_early_warning(df_gsii, featured, best_model)
warning.to_csv(PROJECT_ROOT / TABLES_DIR / "early_warning_status.csv", index=False)
warning

## 13. Lokasi Output

In [ ]:
print("Processed:", PROJECT_ROOT / PROCESSED_DIR)
print("Tables:", PROJECT_ROOT / TABLES_DIR)
print("Figures:", PROJECT_ROOT / FIGURES_DIR)